# Improving Standalone Positioning, SEPT0640 Open Sky Dataset

This notebook reproduces the step by step implementation used in the project for the open sky dataset `SEPT0640.26O`.

It runs the main iterations:

1. Baseline L1 Single Point Positioning
2. Weighted Least Squares using soft elevation weighting
3. L1/L2 ionosphere free positioning
4. Hatch filtered ionosphere free positioning
5. Final method selection based on RMS and stability

The notebook saves only the essential plots needed for the report in `report_figures_SEPT0640`.

Before running, keep this notebook in the same folder as:

1. `rinexReader.py`
2. `SatOrbits.py`
3. `SEPT0640.26O`
4. `COD0OPSRAP_20260640000_01D_05M_ORB.SP3`

In [ ]:
# If needed, install only once:
# !pip install numpy pandas matplotlib

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rinexReader as rr
import SatOrbits as so

## 1. User settings

In [ ]:
BASE_DIR = os.getcwd()

# If needed, use your full Windows path instead:
# BASE_DIR = r"d:\M.Sc. Autonomous Systems - DTU\Spring Semester\30554 GNSS\Lab\lab6"

RINEX_FILE = os.path.join(BASE_DIR, "SEPT0640.26O")
SP3_FILE = os.path.join(BASE_DIR, "COD0OPSRAP_20260640000_01D_05M_ORB.SP3")

FIG_DIR = os.path.join(BASE_DIR, "report_figures_SEPT0640")
CSV_DIR = os.path.join(BASE_DIR, "report_outputs_SEPT0640")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

clight = 299792458.0

L1_CODE = "C1C"
L2_CODE = "C2W"
L1_PHASE = "L1C"
L2_PHASE = "L2W"

consts = ["G"]
sigTypes = [L1_CODE, L2_CODE, L1_PHASE, L2_PHASE]

F1 = 1575.42e6
F2 = 1227.60e6
LAMBDA_L1 = clight / F1
LAMBDA_L2 = clight / F2

x0 = np.array([0.0, 0.0, 0.0, 0.0])

MIN_WEIGHT = 1e-4
SOFT_WEIGHT_FLOOR = 0.3

HATCH_WINDOW = 30
HATCH_RESET_THRESHOLD_M = 20.0

print("RINEX:", RINEX_FILE)
print("SP3:", SP3_FILE)
print("Figure output:", FIG_DIR)
print("CSV output:", CSV_DIR)

## 2. Helper functions

In [ ]:
def read_approx_position_from_rinex(filepath):
    with open(filepath, "r", errors="ignore") as f:
        for line in f:
            if "APPROX POSITION XYZ" in line:
                values = line[:60].split()
                return np.array([float(values[0]), float(values[1]), float(values[2])], dtype=float)
    raise ValueError("APPROX POSITION XYZ not found in RINEX header.")


def ecef_to_geodetic(x, y, z):
    a = 6378137.0
    f = 1.0 / 298.257223563
    e2 = f * (2.0 - f)
    lon = np.arctan2(y, x)
    p = np.sqrt(x**2 + y**2)
    lat = np.arctan2(z, p * (1.0 - e2))
    for _ in range(10):
        N = a / np.sqrt(1.0 - e2 * np.sin(lat)**2)
        h = p / np.cos(lat) - N
        lat_new = np.arctan2(z, p * (1.0 - e2 * N / (N + h)))
        if abs(lat_new - lat) < 1e-12:
            lat = lat_new
            break
        lat = lat_new
    N = a / np.sqrt(1.0 - e2 * np.sin(lat)**2)
    h = p / np.cos(lat) - N
    return lat, lon, h


def ecef_to_enu_matrix(ref_xyz):
    lat, lon, _ = ecef_to_geodetic(ref_xyz[0], ref_xyz[1], ref_xyz[2])
    slat, clat = np.sin(lat), np.cos(lat)
    slon, clon = np.sin(lon), np.cos(lon)
    return np.array([
        [-slon,          clon,          0.0],
        [-slat * clon,  -slat * slon,   clat],
        [ clat * clon,   clat * slon,   slat]
    ])


def ecef_delta_to_enu(delta_xyz, ref_xyz):
    R = ecef_to_enu_matrix(ref_xyz)
    if delta_xyz.ndim == 1:
        return R @ delta_xyz
    return (R @ delta_xyz.T).T


def compute_satellite_elevations(satpos, receiver_xyz):
    sat_xyz = satpos.iloc[:, :3].to_numpy(dtype=float)
    rec_xyz = np.asarray(receiver_xyz, dtype=float)
    los_ecef = sat_xyz - rec_xyz[None, :]
    enu = ecef_delta_to_enu(los_ecef, rec_xyz)
    horizontal = np.sqrt(enu[:, 0]**2 + enu[:, 1]**2)
    return np.arctan2(enu[:, 2], horizontal)


def weights_soft(elevation, floor=SOFT_WEIGHT_FLOOR):
    elevation = np.clip(elevation, 0.0, np.pi / 2.0)
    weights = floor + (1.0 - floor) * np.sin(elevation)
    return np.maximum(weights, MIN_WEIGHT)


def ionosphere_free_combination(v1, v2):
    f1_sq = F1**2
    f2_sq = F2**2
    return (f1_sq * v1 - f2_sq * v2) / (f1_sq - f2_sq)


def create_kernel(obs, satpos, x):
    x = np.asarray(x, dtype=float)
    dx = satpos.iloc[:, 0].to_numpy(dtype=float) - x[0]
    dy = satpos.iloc[:, 1].to_numpy(dtype=float) - x[1]
    dz = satpos.iloc[:, 2].to_numpy(dtype=float) - x[2]
    rng = np.sqrt(dx**2 + dy**2 + dz**2)
    unit_vec = np.column_stack((dx / rng, dy / rng, dz / rng))
    A = np.hstack((-unit_vec, np.ones((len(unit_vec), 1))))
    L = obs.iloc[:, 0].to_numpy(dtype=float) - rng - x[3]
    return L, A


def solve_ls(A, L, weights=None):
    if weights is None:
        N = A.T @ A
        u = A.T @ L
    else:
        weights = np.asarray(weights, dtype=float)
        weights = np.maximum(weights, MIN_WEIGHT)
        W = np.diag(weights)
        N = A.T @ W @ A
        u = A.T @ W @ L
    try:
        return np.linalg.solve(N, u)
    except np.linalg.LinAlgError:
        return np.linalg.lstsq(N, u, rcond=None)[0]


def spp_solver(obs, satpos, x0, weights=None):
    tol = 0.001
    maxiter = 50
    x = np.asarray(x0, dtype=float).copy()
    h = np.array([100.0, 100.0, 100.0])
    curiter = 0
    while np.sum(np.abs(h)) > tol and curiter < maxiter:
        L, A = create_kernel(obs, satpos, x)
        dx = solve_ls(A, L, weights=weights)
        x = x + dx
        h = dx[:3]
        curiter += 1
    return pd.Series(x, index=["X", "Y", "Z", "cdt"])


def hatch_update_for_satellite(sat_id, code_m, phase_m, hatch_state,
                               window=HATCH_WINDOW,
                               reset_threshold=HATCH_RESET_THRESHOLD_M):
    if sat_id not in hatch_state:
        hatch_state[sat_id] = {
            "P_smooth": code_m,
            "P_prev": code_m,
            "Phi_prev": phase_m,
            "count": 1
        }
        return code_m, 1

    state = hatch_state[sat_id]
    P_prev = state["P_prev"]
    Phi_prev = state["Phi_prev"]
    P_smooth_prev = state["P_smooth"]
    count_prev = state["count"]

    code_change = code_m - P_prev
    phase_change = phase_m - Phi_prev

    if abs(code_change - phase_change) > reset_threshold:
        hatch_state[sat_id] = {
            "P_smooth": code_m,
            "P_prev": code_m,
            "Phi_prev": phase_m,
            "count": 1
        }
        return code_m, 1

    count = min(count_prev + 1, window)
    alpha = 1.0 / count
    P_smooth = alpha * code_m + (1.0 - alpha) * (P_smooth_prev + phase_change)

    hatch_state[sat_id] = {
        "P_smooth": P_smooth,
        "P_prev": code_m,
        "Phi_prev": phase_m,
        "count": count
    }
    return P_smooth, 0


def apply_hatch_filter_epoch(obs_if, phase_if_m, hatch_state):
    smoothed_values = []
    reset_count = 0
    for sat_id in obs_if.index:
        code_m = float(obs_if.loc[sat_id, "P_IF"])
        phase_m = float(phase_if_m.loc[sat_id])
        P_smooth, reset_flag = hatch_update_for_satellite(
            sat_id=sat_id,
            code_m=code_m,
            phase_m=phase_m,
            hatch_state=hatch_state
        )
        smoothed_values.append(P_smooth)
        reset_count += reset_flag
    obs_hatch_if = pd.DataFrame(smoothed_values, index=obs_if.index, columns=["P_IF_HATCH"])
    return obs_hatch_if, reset_count


def solution_dict_to_dataframe(sol_dict):
    df = pd.DataFrame(sol_dict).T
    if "X" not in df.columns and "X" in df.index:
        df = df.T
    df = df[["X", "Y", "Z", "cdt"]]
    df.index = pd.to_datetime(df.index, errors="coerce")
    df = df[~df.index.isna()]
    return df.sort_index()


def compute_error_dataframe(solution_df, ref_xyz):
    xyz = solution_df[["X", "Y", "Z"]].to_numpy(dtype=float)
    delta_xyz = xyz - ref_xyz[None, :]
    enu = ecef_delta_to_enu(delta_xyz, ref_xyz)
    err = pd.DataFrame(index=solution_df.index)
    err["dX"] = delta_xyz[:, 0]
    err["dY"] = delta_xyz[:, 1]
    err["dZ"] = delta_xyz[:, 2]
    err["E"] = enu[:, 0]
    err["N"] = enu[:, 1]
    err["U"] = enu[:, 2]
    err["horizontal_error"] = np.sqrt(err["E"]**2 + err["N"]**2)
    err["vertical_error"] = np.abs(err["U"])
    err["3d_error"] = np.sqrt(err["E"]**2 + err["N"]**2 + err["U"]**2)
    return err


def rms(values):
    values = np.asarray(values, dtype=float)
    return np.sqrt(np.nanmean(values**2))


def summarize_errors(error_results):
    rows = []
    for method, err in error_results.items():
        rows.append({
            "method": method,
            "epochs": len(err),
            "mean_horizontal_error_m": np.nanmean(err["horizontal_error"]),
            "rms_horizontal_error_m": rms(err["horizontal_error"]),
            "mean_vertical_error_m": np.nanmean(err["vertical_error"]),
            "rms_vertical_error_m": rms(err["vertical_error"]),
            "rms_3d_error_m": rms(err["3d_error"]),
            "std_E_m": np.nanstd(err["E"]),
            "std_N_m": np.nanstd(err["N"]),
            "std_U_m": np.nanstd(err["U"]),
            "max_horizontal_error_m": np.nanmax(err["horizontal_error"]),
            "max_3d_error_m": np.nanmax(err["3d_error"])
        })
    return pd.DataFrame(rows).set_index("method")


def save_fig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, dpi=250, bbox_inches="tight")
    print("Saved:", path)

## 3. Process SEPT0640 and compute all implementation stages

In [ ]:
ref_xyz = read_approx_position_from_rinex(RINEX_FILE)
print("Reference ECEF position from RINEX header:")
print(ref_xyz)

rinexFile = rr.rinexReader(RINEX_FILE)
svpos = so.sp3Orbits(SP3_FILE)

print("\nReading observations:", sigTypes)
rinexFile.readFile(consts, sigTypes)

solutions = {
    "OLS_L1": {},
    "WLS_soft_L1": {},
    "OLS_IF": {},
    "HATCH_IF": {}
}

last_x = {
    "OLS_L1": x0.copy(),
    "WLS_soft_L1": x0.copy(),
    "OLS_IF": x0.copy(),
    "HATCH_IF": x0.copy()
}

hatch_state = {}
info = {}

startrun = time.time()
print("Computing all methods for SEPT0640: ", end="")

for epoch in rinexFile.timelist:
    print(".", end="")
    obs = rinexFile.get_epoch_data(epoch, oTypes=sigTypes)
    obs = obs.dropna()

    if len(obs) < 4:
        continue

    tau = obs[L1_CODE] / clight
    satpos_full = svpos.getSvPos(epoch, tau)

    if len(satpos_full) < 4 or len(satpos_full) != len(obs):
        continue

    cdts = satpos_full.iloc[:, 3] * clight
    satpos = satpos_full.iloc[:, :3]

    # 1. OLS L1
    obs_l1 = obs[[L1_CODE]].copy()
    obs_l1_corr = obs_l1.copy()
    obs_l1_corr.iloc[:, :] = obs_l1_corr.to_numpy(dtype=float) + cdts.values[:, None]

    x_ols_l1 = spp_solver(obs_l1_corr, satpos, last_x["OLS_L1"], weights=None)
    solutions["OLS_L1"][epoch] = x_ols_l1
    last_x["OLS_L1"] = x_ols_l1.to_numpy(dtype=float)

    # 2. WLS soft L1
    elevation_l1 = compute_satellite_elevations(
        satpos,
        x_ols_l1[["X", "Y", "Z"]].to_numpy(dtype=float)
    )
    w_soft = weights_soft(elevation_l1)
    x_wls_l1 = spp_solver(
        obs_l1_corr,
        satpos,
        x_ols_l1.to_numpy(dtype=float),
        weights=w_soft
    )
    solutions["WLS_soft_L1"][epoch] = x_wls_l1
    last_x["WLS_soft_L1"] = x_wls_l1.to_numpy(dtype=float)

    # 3. Ionosphere free
    p1 = obs[L1_CODE].to_numpy(dtype=float)
    p2 = obs[L2_CODE].to_numpy(dtype=float)
    p1_corr = p1 + cdts.to_numpy(dtype=float)
    p2_corr = p2 + cdts.to_numpy(dtype=float)
    p_if = ionosphere_free_combination(p1_corr, p2_corr)
    obs_if = pd.DataFrame(data=p_if, index=obs.index, columns=["P_IF"])

    x_ols_if = spp_solver(obs_if, satpos, last_x["OLS_IF"], weights=None)
    solutions["OLS_IF"][epoch] = x_ols_if
    last_x["OLS_IF"] = x_ols_if.to_numpy(dtype=float)

    # 4. Hatch IF
    phi1_m = obs[L1_PHASE].to_numpy(dtype=float) * LAMBDA_L1
    phi2_m = obs[L2_PHASE].to_numpy(dtype=float) * LAMBDA_L2
    phi1_corr_m = phi1_m + cdts.to_numpy(dtype=float)
    phi2_corr_m = phi2_m + cdts.to_numpy(dtype=float)
    phi_if_m = ionosphere_free_combination(phi1_corr_m, phi2_corr_m)
    phase_if = pd.Series(data=phi_if_m, index=obs.index, name="PHI_IF")

    obs_hatch_if, reset_count = apply_hatch_filter_epoch(
        obs_if=obs_if,
        phase_if_m=phase_if,
        hatch_state=hatch_state
    )

    if len(obs_hatch_if) >= 4:
        x_hatch_if = spp_solver(obs_hatch_if, satpos, last_x["HATCH_IF"], weights=None)
        solutions["HATCH_IF"][epoch] = x_hatch_if
        last_x["HATCH_IF"] = x_hatch_if.to_numpy(dtype=float)

    info[epoch] = pd.Series({
        "num_sats": len(obs),
        "mean_elev_deg": np.nanmean(np.rad2deg(elevation_l1)),
        "min_elev_deg": np.nanmin(np.rad2deg(elevation_l1)),
        "mean_weight_soft": np.nanmean(w_soft),
        "hatch_resets": reset_count
    })

print("")
print(f"Computed all methods in {round(time.time() - startrun, 3)} seconds")

solution_results = {}
for method, sol_dict in solutions.items():
    if len(sol_dict) > 0:
        solution_results[method] = solution_dict_to_dataframe(sol_dict)
        solution_results[method].to_csv(os.path.join(CSV_DIR, f"solution_{method}_SEPT0640.csv"))

info_df = pd.DataFrame(info).T
info_df.index = pd.to_datetime(info_df.index, errors="coerce")
info_df = info_df[~info_df.index.isna()].sort_index()
info_df.to_csv(os.path.join(CSV_DIR, "processing_info_SEPT0640.csv"))

error_results = {}
for method, df in solution_results.items():
    error_results[method] = compute_error_dataframe(df, ref_xyz)
    error_results[method].to_csv(os.path.join(CSV_DIR, f"errors_{method}_SEPT0640.csv"))

summary_table = summarize_errors(error_results)
summary_table.to_csv(os.path.join(CSV_DIR, "method_error_summary_SEPT0640.csv"))
summary_table.round(3)

## 4. Method selection summary

In [ ]:
main_summary = summary_table.loc[[m for m in ['OLS_L1', 'WLS_soft_L1', 'OLS_IF', 'HATCH_IF'] if m in summary_table.index]]
main_summary.round(3)

## 5. Essential report plots

In [ ]:
# Plot 1: Baseline ENU error
err = error_results["OLS_L1"]
plt.figure(figsize=(10, 5))
plt.plot(err.index, err["E"], label="East error")
plt.plot(err.index, err["N"], label="North error")
plt.plot(err.index, err["U"], label="Up error")
plt.xlabel("Epoch")
plt.ylabel("Error [m]")
plt.title("Baseline OLS L1 ENU Errors")
plt.grid(True)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
save_fig("01_baseline_ols_l1_enu_errors.png")
plt.show()

# Plot 2: WLS horizontal error comparison
plt.figure(figsize=(10, 5))
plt.plot(error_results["OLS_L1"].index, error_results["OLS_L1"]["horizontal_error"], label="OLS L1")
plt.plot(error_results["WLS_soft_L1"].index, error_results["WLS_soft_L1"]["horizontal_error"], label="WLS soft L1")
plt.xlabel("Epoch")
plt.ylabel("Horizontal error [m]")
plt.title("WLS Test: Horizontal Error Comparison")
plt.grid(True)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
save_fig("02_wls_horizontal_error_comparison.png")
plt.show()

# Plot 3: Ionosphere free horizontal and vertical comparison
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True)
ax[0].plot(error_results["OLS_L1"].index, error_results["OLS_L1"]["horizontal_error"], label="OLS L1")
ax[0].plot(error_results["OLS_IF"].index, error_results["OLS_IF"]["horizontal_error"], label="OLS IF")
ax[0].set_title("Horizontal error")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Error [m]")
ax[0].grid(True)
ax[0].legend()

ax[1].plot(error_results["OLS_L1"].index, error_results["OLS_L1"]["vertical_error"], label="OLS L1")
ax[1].plot(error_results["OLS_IF"].index, error_results["OLS_IF"]["vertical_error"], label="OLS IF")
ax[1].set_title("Vertical error")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Error [m]")
ax[1].grid(True)
ax[1].legend()

for a in ax:
    for label in a.get_xticklabels():
        label.set_rotation(45)
fig.suptitle("Effect of L1/L2 Ionosphere Free Correction")
fig.tight_layout()
save_fig("03_ionosphere_free_error_comparison.png")
plt.show()

# Plot 4: Hatch IF horizontal and vertical comparison
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True)
ax[0].plot(error_results["OLS_IF"].index, error_results["OLS_IF"]["horizontal_error"], label="OLS IF")
ax[0].plot(error_results["HATCH_IF"].index, error_results["HATCH_IF"]["horizontal_error"], label="HATCH IF")
ax[0].set_title("Horizontal error")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Error [m]")
ax[0].grid(True)
ax[0].legend()

ax[1].plot(error_results["OLS_IF"].index, error_results["OLS_IF"]["vertical_error"], label="OLS IF")
ax[1].plot(error_results["HATCH_IF"].index, error_results["HATCH_IF"]["vertical_error"], label="HATCH IF")
ax[1].set_title("Vertical error")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Error [m]")
ax[1].grid(True)
ax[1].legend()

for a in ax:
    for label in a.get_xticklabels():
        label.set_rotation(45)
fig.suptitle("Effect of Hatch Filtering on Ionosphere Free Solution")
fig.tight_layout()
save_fig("04_hatch_if_error_comparison.png")
plt.show()

# Plot 5: Final HATCH IF ENU error
err = error_results["HATCH_IF"]
plt.figure(figsize=(10, 5))
plt.plot(err.index, err["E"], label="East error")
plt.plot(err.index, err["N"], label="North error")
plt.plot(err.index, err["U"], label="Up error")
plt.xlabel("Epoch")
plt.ylabel("Error [m]")
plt.title("Final Selected Method: HATCH IF ENU Errors")
plt.grid(True)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
save_fig("05_final_hatch_if_enu_errors.png")
plt.show()

# Plot 6: Satellite availability and Hatch reset count
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True)
ax[0].plot(info_df.index, info_df["num_sats"], label="Number of satellites")
ax[0].set_title("Satellite availability")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Satellite count")
ax[0].grid(True)
ax[0].legend()

ax[1].plot(info_df.index, info_df["hatch_resets"], label="Hatch resets")
ax[1].set_title("Hatch filter resets")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Reset count")
ax[1].grid(True)
ax[1].legend()

for a in ax:
    for label in a.get_xticklabels():
        label.set_rotation(45)
fig.suptitle("Data Continuity Indicators")
fig.tight_layout()
save_fig("06_satellite_and_hatch_reset_summary.png")
plt.show()

## 6. Optional coordinate plots

In [ ]:
for coord in ["X", "Y", "Z"]:
    plt.figure(figsize=(10, 5))
    for method in ["OLS_L1", "WLS_soft_L1", "OLS_IF", "HATCH_IF"]:
        if method in solution_results:
            plt.plot(solution_results[method].index, solution_results[method][coord], label=method)
    plt.xlabel("Epoch")
    plt.ylabel(f"{coord} [m]")
    plt.title(f"Receiver {coord} Coordinate Comparison")
    plt.grid(True)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    save_fig(f"optional_{coord}_coordinate_comparison.png")
    plt.show()

## 7. Final selection

The final selected method is `HATCH_IF`.

The reason is not that it magically removes all errors. It wins because it combines the two most useful improvements:

1. The ionosphere free combination reduces a real physical bias.
2. Hatch filtering reduces short term noise and large jumps.

WLS was kept as an important tested method, but it was not the main winner for the open sky dataset.